In [1]:
# Enable Float64 for more stable matrix inversions.
from jax import config
import jax
import jax.numpy as jnp
import jax.random as jr
from jaxtyping import install_import_hook
import matplotlib.pyplot as plt

config.update("jax_enable_x64", True)

with install_import_hook("gpjax", "beartype.beartype"):
    import gpjax as gpx

key = jr.key(123)

from gpkanmodel.model import GPKAN

from gpjax.parameters import (
    transform,
    DEFAULT_BIJECTION,
    Parameter,
    PositiveReal,
    Real,
)

/Users/aaron/Documents/masters/gpkan/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = GPKAN(init_paramters=[2.23, 2.34155])
params = model.kernel_parameters

Model initialized.


In [3]:
params0 = params[0][0][0]
params0

State({
  'likelihood': {
    'obs_stddev': VariableState(
      type=PositiveReal,
      value=Array(1., dtype=float64, weak_type=True),
      _tag='positive'
    )
  },
  'prior': {
    'kernel': {
      'lengthscale': VariableState(
        type=PositiveReal,
        value=Array(2.23, dtype=float64, weak_type=True),
        _tag='positive'
      ),
      'variance': VariableState(
        type=PositiveReal,
        value=Array(2.34155, dtype=float64, weak_type=True),
        _tag='positive'
      )
    },
    'mean_function': {
      'constant': VariableState(
        type=Real,
        value=Array(0., dtype=float64, weak_type=True),
        _tag='real'
      )
    }
  }
})

In [4]:
def softplus(x):
    """Forward transformation: ensures positivity"""
    return jnp.log(jnp.exp(x) + 1.0)

def softplus_inv(x):
    """Inverse softplus: transforms from constrained to unconstrained space"""
    return jnp.log(jnp.exp(x) - 1.0)


    # return jax.tree_util.tree_map_with_path(apply_transform, states_list)

In [5]:
import jax
import jax.numpy as jnp
from flax import nnx

# Your definitions
softplus = lambda x: jnp.log(1.0 + jnp.exp(x))
inv_softplus = lambda x: jnp.log(jnp.exp(x) - 1.0)

def transform_state(state_tree, transform_fn):
    def map_fn(path, leaf):
        # path is a tuple of keys, e.g., ('prior', 'kernel', 'lengthscale')
        # We check if the leaf is a VariableState and if the name matches
        if isinstance(leaf, nnx.VariableState):
            # Option A: Target by key name
            # name = path[-1].key if hasattr(path[-1], 'key') else str(path[-1])
            # if name in ('lengthscale', 'variance'):
            #     return leaf.replace(value=transform_fn(leaf.value))
            
            # Option B: Target by tag (uncomment if preferred)
            if getattr(leaf, '_tag', None) == 'positive':
                return leaf.replace(value=transform_fn(leaf.value))
                
        return leaf

    return jax.tree_util.tree_map_with_path(map_fn, state_tree)

# Since you have a list of lists, we wrap it in another tree_map
transformed_nested_list = jax.tree_util.tree_map(
    lambda s: transform_state(s, softplus),
    params
)

## manual

In [6]:
# for i, layer in enumerate(params):
#     for j, in_dim in enumerate(layer):
#         for k, out_neuron in enumerate(in_dim):
#             params[i][j][k] = transform(out_neuron, DEFAULT_BIJECTION, inverse=True)


In [7]:
gamga = jax.tree_util.tree_map(
    lambda s: transform(s, DEFAULT_BIJECTION, inverse=True),
    params,
    is_leaf=lambda x: isinstance(x, nnx.State)
)

In [11]:
jax.tree_util.tree_map(
    lambda s: transform(s, DEFAULT_BIJECTION, inverse=False),
    gamga,
    is_leaf=lambda x: isinstance(x, nnx.State)
)

[[[State({
     'likelihood': {
       'obs_stddev': VariableState(
         type=PositiveReal,
         value=Array(1., dtype=float64, weak_type=True),
         _tag='positive'
       )
     },
     'prior': {
       'kernel': {
         'lengthscale': VariableState(
           type=PositiveReal,
           value=Array(2.23, dtype=float64, weak_type=True),
           _tag='positive'
         ),
         'variance': VariableState(
           type=PositiveReal,
           value=Array(2.34155, dtype=float64, weak_type=True),
           _tag='positive'
         )
       },
       'mean_function': {
         'constant': VariableState(
           type=Real,
           value=Array(0., dtype=float64, weak_type=True),
           _tag='real'
         )
       }
     }
   }),
   State({
     'likelihood': {
       'obs_stddev': VariableState(
         type=PositiveReal,
         value=Array(1., dtype=float64, weak_type=True),
         _tag='positive'
       )
     },
     'prior': {
       'kern

In [8]:
gamga

[[[State({
     'likelihood': {
       'obs_stddev': VariableState(
         type=PositiveReal,
         value=Array(0.54132485, dtype=float64),
         _tag='positive'
       )
     },
     'prior': {
       'kernel': {
         'lengthscale': VariableState(
           type=PositiveReal,
           value=Array(2.11623938, dtype=float64),
           _tag='positive'
         ),
         'variance': VariableState(
           type=PositiveReal,
           value=Array(2.24042667, dtype=float64),
           _tag='positive'
         )
       },
       'mean_function': {
         'constant': VariableState(
           type=Real,
           value=Array(0., dtype=float64, weak_type=True),
           _tag='real'
         )
       }
     }
   }),
   State({
     'likelihood': {
       'obs_stddev': VariableState(
         type=PositiveReal,
         value=Array(0.54132485, dtype=float64),
         _tag='positive'
       )
     },
     'prior': {
       'kernel': {
         'lengthscale': Variable